# Week 4 · Functions you can trust · Your first files · Midterm

[![Open in Colab](../assets/badges/colab-badge.svg)](https://colab.research.google.com/github/loem-ms/programming-for-information-systems-site/blob/main/content/downloads/Week_04.ipynb)

Your functions from Weeks 2 and 3 leave the notebook this week. They become files in **your own system folder**, which grows every week until Week 10.

| | |
|---|---|
| **This week's question** | How do you make a function people can trust, and where does it live in a real system? |
| **Python you will learn** | parameters, `return` vs `print`, `None`, docstrings, flowchart → function, `assert`, test functions, creating `.py` files |
| **Principle 4** | **One function, one job.** Return values instead of printing them. Test normal, boundary and odd cases. |
| **Files you add** | `my_foodbridge/rules.py` (your Week 2 rules) and `my_foodbridge/records.py` (your Week 3 functions + the new `open_portions`) |
| **What you will see** | `bridge.show()`: the map of your system, with your parts lit up |
| **Session 3** | the individual midterm, in `Midterm.ipynb` |

*Campus FoodBridge is fictional. All organisations, people and numbers are invented.*

![System map, week 4: your first files: rules.py and records.py](../assets/diagrams/system_map_w04.svg)

### The story so far

In Week 2 you wrote the rules `pickup_label` and `urgency`. In Week 3 you wrote `total_portions` and `find_offer`. They work, but they live in old notebooks. Every week you would have to copy them again.

This morning the coordinator asks a new question: *"How many portions are still open?"* A classmate's helper answers by **printing** a number, so no other code can use it.

This week you make functions people can **trust**: a clear promise, one returned answer, and tests that prove it. Then you move them into files. From now on, each week adds one file to your folder, and by Week 10 the folder **is** your system.

## Before you start

1. Click **Copy to Drive** (or **File → Save a copy in Drive**), then work in your copy.
2. Run the setup cell. In Colab it asks to connect Google Drive: say **yes**, so your `my_foodbridge/` folder is still there next week. (If you say no, your files disappear when the runtime stops.)
3. ⭐ = core for everyone, ⭐⭐ = optional stretch.
4. This notebook covers Sessions 1 and 2. Session 3 is the midterm: read the briefing at the end of Session 2.

In [ ]:
# ▶ Run this cell first (setup)
# In Colab or Jupyter, this cell unpacks the FoodBridge kit next to the notebook.
# This web page is for reading. To run the notebook, use the 'Open in Colab' button at the top of this page.


## Session 1 · Learn and try: functions you can trust (90 min)

### The idea: modules and contracts

A phone charger makes a promise. Wall power goes in, safe power for your phone comes out, and it never damages the phone. You do not need to know what is inside. You trust it because it keeps the same promise every time, and because someone tested it.

A function is the same. Its **contract** says three things:

- **Inputs:** what you must give it. `find_offer` needs a list of records and an ID.
- **Output:** what it returns. `find_offer` returns the record, or `None` when there is no such ID.
- **What it never does.** `find_offer` never changes the list you give it, and never prints.

A **module** is a file of functions with contracts. When every part keeps its contract, you can build a system from parts, and replace one part (with your own function, or one an AI wrote) without breaking the others. A **test** is how you *show* that a part keeps its contract.

### 1 · Parameters and arguments

**Parameters** are the names in the `def` line: the function's inputs. **Arguments** are the values you give when you call it. Python matches them by position: the first argument goes to the first parameter. You can also give them by name.

🤔 **Predict** the last line before you run the cell.

In [ ]:
def price_after_discount(price_riel, discount_percent):
    return price_riel - price_riel * discount_percent // 100

print(price_after_discount(10000, 10))
print(price_after_discount(discount_percent=25, price_riel=4000))   # by name: order does not matter
print(price_after_discount(10, 10000))                              # 🤔 arguments swapped

**Prediction:** …  **Result:** …

A price of −990 riel is nonsense, but Python does not know that. Only a person, or a test, notices. Names with units (`price_riel`) make swapped arguments easier to spot.

### ✍️ Try it 1 · `phone_bill`

A phone plan charges for every minute. Write `phone_bill(minutes, riel_per_minute)`: return the cost in riel.

In [ ]:
def phone_bill(minutes, riel_per_minute):
    # TODO: return the cost
    return None

bridge.check("phone_bill", phone_bill, examples=[
    ((10, 50), 500),
    ((0, 50), 0),
    ((3, 120), 360),
])

Each example is *(inputs, expected output)*. With two inputs, the inputs go in brackets: `(10, 50)`. The examples **are** the contract, written so a computer can check it.

### 2 · `return` or `print`?

Both functions below add up the same prices. Only one gives the answer back.

In [ ]:
def total_printed(prices_riel):
    total = 0
    for price_riel in prices_riel:
        total += price_riel
    print(total)

def total_returned(prices_riel):
    total = 0
    for price_riel in prices_riel:
        total += price_riel
    return total

printed_result = total_printed([4000, 1500])
returned_result = total_returned([4000, 1500])
print("printed_result is", printed_result)
print("returned_result is", returned_result)

- `print` **shows** a value to a person. Then it is gone.
- `return` **gives** the value back to the code that called the function. That code can add it, compare it, test it or put it on a screen.

The system never reads the screen. A function that only prints is useless to the rest of the system, and impossible to test.

### ✍️ Try it 2 · Return, don't print

The coordinator's screen should say *"Send a volunteer now"* when at least one offer is urgent. This function counts urgent offers, but nothing can use its answer. Run the cell and read the 💡 hint. Then fix the function.

In [ ]:
URGENT_HOURS = 3

def count_urgent(hours_left_list):
    count = 0
    for hours_left in hours_left_list:
        if hours_left <= URGENT_HOURS:
            count += 1
    print(count)

bridge.check("count_urgent", count_urgent, examples=[
    ([2, 5, 3, 20], 2),
    ([12, 24], 0),
    ([], 0),
])

<details><summary>Show a possible answer</summary>

Replace `print(count)` with `return count`. If a person also wants to see the number, print it **outside** the function: `print(count_urgent([2, 5, 3, 20]))`.

</details>

### 3 · `None` means "no answer"

A function that ends without `return` gives back `None`. That is what happened to `printed_result`. Sometimes `None` is **on purpose**: `find_offer` returns `None` to say "not found". Check for it with `is None`.

In [ ]:
prices_by_item = {"notebook": 4000, "pen": 1500}

def find_price(item_name):
    return prices_by_item.get(item_name)

price = find_price("ruler")
if price is None:
    print("Not sold here")
else:
    print(price, "riel")

`None` on purpose is part of the contract, so write it down. `None` by accident (a forgotten `return`) is a bug. (You can also pass a second value to `.get`, such as `prices_by_item.get("ruler", 0)`, to get a default instead of `None`.)

### ✍️ Try it 3 · `find_phone`

Write `find_phone(contacts, name)`: return the phone number for that name, or `None` when the name is not in `contacts`. Do not print anything.

In [ ]:
def find_phone(contacts, name):
    # TODO: return the number, or None
    return "?"

classmates = {"Dara": "012 345 678", "Sokha": "097 111 222"}
bridge.check("find_phone", find_phone, examples=[
    ((classmates, "Dara"), "012 345 678"),
    ((classmates, "Vanna"), None),
    (({}, "Dara"), None),
])

### 4 · Docstrings: write the promise down

A **docstring** is text in triple quotes on the first line inside a function. It states the contract. `help(...)` shows it. The kit's functions have docstrings too.

In [ ]:
def total_price(prices_riel):
    """Return the total of a list of prices in riel. Returns 0 for an empty list."""
    total = 0
    for price_riel in prices_riel:
        total += price_riel
    return total

help(total_price)
help(bridge.records.find_offer)

### ✍️ Try it 4 · `average_mark`

Write `average_mark(marks)` with a docstring. The promise: return the average of the marks, and **0 for an empty list** (a class with no marks yet must not crash the report).

In [ ]:
def average_mark(marks):
    """TODO: write the promise here."""
    # TODO
    return None

bridge.check("average_mark", average_mark, examples=[
    ([60, 80], 70),
    ([100], 100),
    ([0, 100], 50),
    ([], 0),
])

<details><summary>Show the trap</summary>

Without the `if len(marks) == 0` line, the empty list stops with `ZeroDivisionError`. The empty case is exactly the case a docstring should mention, and a test should check.

</details>

### 5 · From a flowchart to a function

Before you write a rule, draw it. A **flowchart** uses three shapes: an oval for the start, a diamond for a yes/no question, and a box for an end. Here is your Week 2 pickup rule:

![Flowchart of pickup_label: not approved gives hold; otherwise team at or above the team minimum, else standard](../assets/diagrams/flowchart_pickup.svg)

Each **diamond** becomes an `if`. Each **end box** becomes a `return`. Follow every arrow once, and you have tested every path.

In [ ]:
TEAM_PICKUP_MIN_PORTIONS = 20

def pickup_label(portions, approved):
    if not approved:                              # diamond 1
        return "hold"
    if portions >= TEAM_PICKUP_MIN_PORTIONS:      # diamond 2
        return "team"
    return "standard"

print(pickup_label(30, False), pickup_label(20, True), pickup_label(19, True))

### ✍️ Try it 5 · `bus_fare` from a flowchart

A city bus has three fares:

```text
       ( start )
           │
  ◇ age ≤ 5? ◇ ── yes ──▶ [ 0 riel ]
           │ no
  ◇ age ≥ 60? ◇ ── yes ──▶ [ 1000 riel ]
           │ no
     [ 2000 riel ]
```

Write `bus_fare(age)` from the flowchart. The examples test every arrow, and the **boundaries** (5/6 and 60).

In [ ]:
CHILD_MAX_AGE = 5
SENIOR_MIN_AGE = 60

def bus_fare(age):
    # TODO: one if for each diamond, one return for each end box
    return None

bridge.check("bus_fare", bus_fare, examples=[
    (5, 0),
    (6, 2000),
    (59, 2000),
    (60, 1000),
])

### 6 · Build from small steps: rounding up

Some answers must be **whole things**: buses, baskets, volunteers. Break the problem into small steps. `//` gives the number of **full** groups. `%` gives what is **left over**. If anything is left over, you need one more.

🤔 **Predict** both lines.

In [ ]:
BOXES_PER_BAG = 6
print(14 // BOXES_PER_BAG, 14 % BOXES_PER_BAG)
print(12 // BOXES_PER_BAG, 12 % BOXES_PER_BAG)

**Prediction:** …  **Result:** …

### ✍️ Try it 6 · `minibuses_needed`

A class trip: each minibus has `SEATS_PER_MINIBUS` seats. Nobody travels in half a minibus. Write `minibuses_needed(students)`.

In [ ]:
SEATS_PER_MINIBUS = 12

def minibuses_needed(students):
    """Return how many minibuses carry this many students. 0 students need 0 minibuses."""
    # TODO: full minibuses with //, then one more if % shows anyone left over
    return None

bridge.check("minibuses_needed", minibuses_needed, examples=[
    (24, 2),
    (25, 3),
    (12, 1),
    (0, 0),
])

<details><summary>Show two traps</summary>

`students / SEATS_PER_MINIBUS` gives `2.083…`, which is not a number of minibuses. `students // SEATS_PER_MINIBUS + 1` gives 3 for exactly 24 students (one bus too many **at the boundary**) and 1 bus for 0 students.

</details>

### 7 · `assert`: a check that stops loudly

`bridge.check` is the kit's checker. You can write checks yourself with `assert`. It checks that something is `True`. If it is, nothing happens. If it is not, Python stops with an `AssertionError`. A broken promise should never pass quietly.

In [ ]:
assert price_after_discount(10000, 10) == 9000
assert total_price([]) == 0
print("Both asserts passed")
# assert total_price([4000, 1500]) == 5000    # remove the # to see a failing assert, then put it back

### 8 · Test functions: normal, boundary and empty

A **test function** has a name that starts with `test_`. It uses `assert` to check one kind of case:

| Kind | The question | For `total_price` |
|---|---|---|
| **normal** | Does it work for a usual input? | `[4000, 1500]` → 5500 |
| **boundary** | Does it work at the edge? | `[4000]` → 4000 |
| **empty or odd** | Missing, empty or unusual input? | `[]` → 0 |

The cell also defines `run_tests`, a tiny test runner. You do not need to write one; just read it. It calls each test and prints ✅ when no `assert` failed, or ❌ with the reason.

In [ ]:
def run_tests(tests):
    """Run each test function. Print ✅ or ❌ for each one; never stop the notebook."""
    passed = 0
    for test in tests:
        try:
            test()
            print("✅", test.__name__)
            passed += 1
        except AssertionError as problem:
            print("❌", test.__name__, "→", problem)
        except Exception as problem:
            print("❌", test.__name__, "→ error:", type(problem).__name__, problem)
    print(passed, "of", len(tests), "tests passed")

def test_total_normal():
    assert total_price([4000, 1500]) == 5500

def test_total_boundary():
    assert total_price([4000]) == 4000

def test_total_empty():
    assert total_price([]) == 0

run_tests([test_total_normal, test_total_boundary, test_total_empty])

Notice `run_tests([test_total_normal, ...])`: you give the **functions** without `( )`. The runner calls them.

### ✍️ Try it 7 · Tests that follow the constant

Write three tests for `minibuses_needed`. Write the amounts **with the constant**: `minibuses_needed(SEATS_PER_MINIBUS + 1)`, not `minibuses_needed(13)`.

In [ ]:
def test_minibuses_normal():
    # TODO: two full minibuses plus one more student need 3 minibuses
    assert False, "write this test"

def test_minibuses_boundary():
    # TODO: exactly one full minibus needs 1; one more student needs 2
    assert False, "write this test"

def test_minibuses_empty():
    # TODO: 0 students need 0 minibuses
    assert False, "write this test"

run_tests([test_minibuses_normal, test_minibuses_boundary, test_minibuses_empty])

Now a bigger minibus is hired. Change `SEATS_PER_MINIBUS` to 15 in Try it 6 and run it (Try it 6's own check will show ❌ for 25 — that is expected, because its examples are for 12 seats), then run your tests again. Do they still pass? Then set it back to 12.

**What happened:** …

<details><summary>Show what should happen</summary>

With the constant, the tests pass for 12 seats and for 15. A test such as `assert minibuses_needed(13) == 2` passes for 12 seats but **fails** for 15. When a test breaks only because a decision changed, it has a hidden **magic number**. (The `bridge.check` examples in Try it 6 use raw numbers, so they do break: they test one decision only.)

</details>

### ⭐⭐ Try it 8 · A test that catches a bug

The docstring and the code below do not agree. Write a test that **fails** because of the bug. Then fix the function so your test passes.

In [ ]:
def count_not_finished(records):
    """Return how many offers still need work: status new, approved or picked_up."""
    count = 0
    for record in records:
        if record["status"] in ["new", "approved"]:
            count += 1
    return count

def test_picked_up_still_needs_work():
    # TODO: one record with status "picked_up" -> the answer should be 1
    assert False, "write this test"

run_tests([test_picked_up_still_needs_work])

**What you can now do:** write a function with a clear contract (inputs, one returned answer, a docstring), draw it as a flowchart first, and prove it keeps its promise with checks and tests for normal, boundary and empty cases. Next: put your own functions where a real system keeps them.

## Session 2 · Your first files (90 min)

*New runtime since Session 1? Use **Runtime → Run before**, so `bridge` and `run_tests` are ready.*

### Why files? How a real system is laid out

A real information system is not one long notebook. It is a **folder of files**, each with one job, so people can read, test, replace or hand over one file without breaking the others. From today you keep **your own folder**, `my_foodbridge/`, and add one file each week:

```text
my_foodbridge/
├── rules.py        ← part: your rules (Week 4)
├── records.py      ← part: totals and search (Week 4)
├── data_model.py   ← part: the check at the door (Week 5)
├── lifecycle.py    ← part: allowed status moves (Week 6)
├── storage.py      ← part: save and load (Week 7)
├── report.py       ← part: the public board (Week 8)
├── app.py          ← entry point: the program a person starts (Week 9)
└── tests.py        ← tests for your parts (Week 10)
```

A system has **two kinds of `.py` file**:

- A **part** (a **module**) holds rules and functions. It **returns** answers, never calls `input()` or `print()`, and other files **import** it.
- An **entry point** is the file a person starts. It imports the parts, talks to the user, and shows the answers.

By Week 10 the folder is your whole system.

**How to create and save a `.py` file in Colab:**

1. Open the **Files** panel (the folder icon on the left; click Refresh if needed).
2. Right-click `my_foodbridge` → **New file** → type the file name (for example `rules.py`).
3. Double-click the file to open it in the editor, paste or write your code, and press **Ctrl+S** (Mac: **Cmd+S**) to save.

*(Running **Run all** never overwrites files you saved in `my_foodbridge/`.)*

### 📁 Your folder now

`bridge.show()` draws the map of your system. Every part you have not written yet runs the kit's **model version**, so the whole system always works. Run it now: everything is ⬜.

In [ ]:
bridge.show()

### 🎫 This week's ticket

> **From:** the coordinator · **To:** you
> "Your rules and record functions are spread over three notebooks. Please put them in our system, where everyone can find and test them. And I need one new number every morning: how many portions are still open?"

| File | Function | Contract | Done when |
|---|---|---|---|
| `rules.py` | `pickup_label(portions, approved)` | `"hold"` / `"team"` / `"standard"` (your Week 2 rule) | its check passes |
| `rules.py` | `urgency(hours_left)` | `"urgent"` / `"soon"` / `"later"` | its check passes |
| `records.py` | `total_portions(records)` | all the portions (100 for the sample) | its check passes |
| `records.py` | `find_offer(records, offer_id)` | the record, or `None` | its check passes |
| `records.py` | `open_portions(records)` **new** | portions with status new, approved or picked_up (42 for the sample) | its check passes, and `bridge.show()` lights 5 parts |

Your Week 2 and Week 3 code is in your copies of those notebooks (in Google Drive). Open them in another tab and copy your functions, or rewrite them from your flowchart.

### ✏️ Model it on paper first

Draw the flowchart for `urgency(hours_left)` on paper: one start, two diamonds (`hours_left ≤ URGENT_HOURS?`, `hours_left ≤ SOON_HOURS?`), three end boxes. Then check: which numbers would test every arrow, and both boundaries?

In [ ]:
# 📝 Describe your flowchart, and list the hours_left values that test every arrow:

### Write `rules.py`

Create `my_foodbridge/rules.py` in the **Files** panel, copy the starter below, set the three constants to **your** Week 2 decisions, paste or rewrite your two rules, and save (**Ctrl+S** / **Cmd+S**).

```python
# rules.py · my Week 2 rules, now in a file
TEAM_PICKUP_MIN_PORTIONS = 20   # this many portions or more needs a team
URGENT_HOURS = 3                # hours left at or below this = urgent
SOON_HOURS = 12                 # hours left at or below this = soon


def pickup_label(portions, approved):
    """Return "hold", "team" or "standard" for one offer."""
    # TODO: your Week 2 rule, using TEAM_PICKUP_MIN_PORTIONS
    return "?"


def urgency(hours_left):
    """Return "urgent", "soon" or "later"."""
    # TODO: your Week 2 rule, using URGENT_HOURS and SOON_HOURS
    return "?"
```

`bridge.check(..., folder="my_foodbridge")` reads the function **and your constants** from the file. The boundary cases come from your constants, so your decision never makes the check fail.

In [ ]:
bridge.check("pickup_label", folder="my_foodbridge")
bridge.check("urgency", folder="my_foodbridge")

### 🧩 Ladder 1 · Put the lines in order

`open_portions` is new. Before you write it, build it from its lines. Click the lines into order (one line does not belong), then press **Check**.

In [ ]:
bridge.parsons("open_portions")

*On the course website the puzzle does not run. Open the notebook in Colab, or write the letters in order in your notes.*

<details><summary>Show the answer</summary>

```python
def open_portions(records):
    total = 0
    for record in records:
        if record["status"] in ["new", "approved", "picked_up"]:
            total = total + record["portions"]
    return total
```

The line that does not belong is `print(total)`. `print` shows the total on the screen but gives nothing back, so the system cannot use it.

</details>

### ⭐⭐ Ladder 2 · Fill the blanks

*Optional practice.* The same pattern, one more time: a total of only **some** records. Fill the two blanks in `delivered_portions`.

In [ ]:
def delivered_portions(records):
    """Return the total portions with status "delivered". Does not change records."""
    total = 0
    for record in records:
        if record["status"] == ____:
            total = total + ____
    return total

bridge.check("delivered_portions", delivered_portions, examples=[
    (bridge.sample_records(), 18),
    ([], 0),
])

### 🧩 Ladder 3 · From a blank file: `records.py`

Create `my_foodbridge/records.py` in the **Files** panel, copy the starter below, write your two Week 3 functions and `open_portions` from scratch, and save (**Ctrl+S** / **Cmd+S**). Open means status `"new"`, `"approved"` or `"picked_up"`: food that still needs volunteers.

```python
# records.py · my Week 3 functions, now in a file


def total_portions(records):
    """Return the total portions in these records."""
    # TODO: your Week 3 accumulator
    return 0


def find_offer(records, offer_id):
    """Return the record with this ID, or None when there is none."""
    # TODO: your Week 3 search
    return None


def open_portions(records):
    """Return the portions still waiting: status new, approved or picked_up."""
    # TODO: new this week
    return 0
```

In [ ]:
bridge.check("total_portions", folder="my_foodbridge")
bridge.check("find_offer", folder="my_foodbridge")
bridge.check("open_portions", folder="my_foodbridge")

### 👀 See it

Run the map again. Every part that passes its check is now **yours** (✅), and the system uses your code for it. A part that does not pass yet shows 🔧, and the model version keeps the system running until you fix it.

In [ ]:
bridge.show()

### 🧭 Your decisions travel with your files

Your Week 2 decisions are now the constants at the top of `rules.py`. Try it: open `my_foodbridge/rules.py` in the **Files** panel, change `URGENT_HOURS` (for example from 3 to 4), save (**Ctrl+S** / **Cmd+S**), then run the cell below. The check still passes, because it reads your constant: it now shows `urgency(4) → 'urgent'`, so F004 (rice meals, 4 hours left) becomes urgent. The map lists your decision. You did not change the check or the map; both follow your file.

In [ ]:
bridge.check("urgency", folder="my_foodbridge")
bridge.show()

**Decision log** — *My choice:* … · *An alternative:* … · *Why I chose it:* … · *Who is affected / the tradeoff:* …

### ⚖️ Good or risky?

A classmate asked an AI assistant for *"a function that gives the delivered portions and cleans up finished offers, with a test."* It wrote **Version A**. **Version B** is a rewrite. We added `_a` and `_b` to the names so both can live in this notebook. Run both cells: every test passes.

In [ ]:
def delivered_portions_a(records):
    total = 0
    for record in records:
        if record["status"] == "delivered":
            total += record["portions"]
            records.remove(record)          # clean up finished offers
    print("Delivered:", total, "portions")

def test_delivered_a():
    delivered_portions_a(bridge.sample_records())

run_tests([test_delivered_a])

In [ ]:
def delivered_portions_b(records):
    """Return the total portions with status "delivered". Does not change records."""
    total = 0
    for record in records:
        if record["status"] == "delivered":
            total += record["portions"]
    return total

def test_delivered_normal():
    assert delivered_portions_b(bridge.sample_records()) == 18

def test_delivered_does_not_change_records():
    records = bridge.sample_records()
    delivered_portions_b(records)
    assert len(records) == 4

run_tests([test_delivered_normal, test_delivered_does_not_change_records])

Both "pass". Find **three** problems in Version A. For each, say which person or record is harmed.

In [ ]:
# 📝 Problem 1:
# 📝 Problem 2:
# 📝 Problem 3:

<details><summary>Show the instructor's view</summary>

1. **It changes the list it is given.** `records.remove(record)` deletes delivered offers from the coordinator's own list. F002 (18 portions, delivered to Riverside Community Kitchen) disappears from every later report: the donor's thank-you letter and the monthly totals now say less food was saved. Removing items while looping over the same list can also skip the next record.
2. **It prints instead of returning.** The report and the tests cannot use the number. `delivered_portions_a(...)` gives back `None`.
3. **Its test checks nothing.** `test_delivered_a` has no `assert`, so it passes even when the answer is wrong. A test that cannot fail is not a test.

Version B has one job, returns its answer, says in its docstring that it does not change the list, and has a test that proves it.

</details>

### ⭐⭐ Optional · An entry point for your rules

*Optional — this section is not checked and is not in the midterm.*

Create `my_foodbridge/run_rules.py` in the **Files** panel, paste the tiny **entry point** below, and save (**Ctrl+S** / **Cmd+S**). It imports **your** `rules.py` (line 2), asks questions and prints the answers. You write your own entry point, `app.py`, in Week 9. (`check` and the map skip `run_` files: they are entry points, not parts.)

```python
# run_rules.py · a tiny program that imports and runs your rules.py
from rules import pickup_label, urgency


def main():
    print("=== Campus FoodBridge · Rule Check ===")
    portions = int(input("Portions? "))
    hours_left = int(input("Hours left? "))
    approved = input("Approved (y/n)? ").strip().lower() == "y"
    print("Pickup:", pickup_label(portions, approved))
    print("Urgency:", urgency(hours_left))


if __name__ == "__main__":
    main()
```

To run it, remove the `#` on the second line below and type `20`, `4`, `y`. Then open `my_foodbridge/rules.py` in the **Files** panel, change `URGENT_HOURS` (3 ↔ 4), save (**Ctrl+S** / **Cmd+S**), and run this cell again with the same answers: the urgency changes, but `run_rules.py` did not. Put the `#` back afterwards, so **Run all** does not wait for input. (This works in Colab. In local Jupyter, use a terminal: `cd my_foodbridge`, then `python3 run_rules.py`.)

*Curious?* Type `many` for portions: the program crashes with `ValueError`. Checking input is next week's job.

In [ ]:
# Remove the # on the next line, run the cell, then type portions, hours left, and y/n.
# !cd my_foodbridge && python -B -u run_rules.py

## The road to Week 10

Every week has the same rhythm: 🤔 predict → 🎫 ticket → ✏️ model on paper → 🧩 ladder → ✍️ make the file → 👀 see it work.

| Week | File you add | What you will see |
|---|---|---|
| 5 | `data_model.py` · `clean_offer` | nine messy rows turn green or red, with your messages |
| 6 | `lifecycle.py` · `can_move` | the state diagram, drawn from your code |
| 7 | `storage.py` · save, load, history | your saved files and a timeline of the day |
| 8 | `report.py` · the public board | the coordinator's screen next to the public board |
| 9 | `app.py` · your own menu | your whole app running, and charts of a whole semester |
| 10 | `tests.py` | a gallery of your system, and a clean run |

**The final project** (Weeks 9–10) is your folder, presented: it runs as an app, with at least three decisions of your own and a decision log (required); an extension with tests, or a screen made with an AI's help, is optional.

## How to ask an AI for code

AI assistants write code quickly. You stay responsible for what it does to people. Two habits make the difference.

**1 · Ask with a contract, not a wish.** Compare:

> ❌ *"Write a function for open portions."*
>
> ✅ *"Write a Python 3 function `open_portions(records)`. `records` is a list of dictionaries with the keys `"status"` and `"portions"` (an int). Return the total portions of the records whose status is `"new"`, `"approved"` or `"picked_up"`. Return 0 for an empty list. Do not print. Do not change the list. Use only plain Python, no libraries. Example: two records, 30 portions picked_up and 18 delivered → 30."*

The second request is this week's ticket, written out: name, inputs, output, odd cases, what it must never do, and an example.

**2 · Check what comes back, as if a stranger wrote it.** Read every line. Run the checks and your own tests. Ask: *what happens with an empty list? at the boundary?* If you cannot explain a line, you cannot use it.

### ⭐⭐ Stretch · Check an AI's answer

An AI assistant answered a vague request with the function below. Run the check. Find the line that breaks the contract, and fix it.

In [ ]:
def ai_open_portions(records):
    # Written by an AI assistant
    total = 0
    for record in records:
        if record["status"] != "rejected":
            total += record["portions"]
    return total

bridge.check("open_portions", ai_open_portions)

<details><summary>Show the problem</summary>

`!= "rejected"` counts everything that is not rejected, so F002's 18 **delivered** portions count as "open", and so would a cancelled offer. The coordinator would send volunteers for food that is already gone. The fix lists the open statuses exactly: a rule written as "everything except…" breaks as soon as a new status appears.

</details>

**The rules for AI in this course:** in weekly work you may use an AI assistant, but you must be able to explain every line you hand in (the instructor may ask you). In the **midterm**, AI tools are not allowed.

## Session 3 · Midterm briefing

Session 3 is the individual midterm: **"Campus Library Desk"**, **60 minutes** (+ 30 minutes setup, submission & live debrief). The notebook is unlocked in class with a **PIN code** from the **Midterm** menu tab.

- **Individual work.** Allowed: your own notes, the weekly notebooks and the Python Basics notebook. Not allowed: AI tools, messaging or sharing code.
- **Four parts (100 points):** 1 Read & trace code by hand (24) · 2 Flowchart → rule function + decision impact (24) · 3 Work with records → summary dictionary (28) · 4 Audit, test & repair AI code + harm report (24) · End: **My key line** (mandatory) + clean run.
- Afterwards: live debrief and a short **2–3 minute oral check** on your own code.

**How to prepare:** redo the 🤔 Predict cells of Weeks 1–4 and explain each answer aloud; flowchart → `if/elif/else` with named constants; loops over records without mutating input; writing `assert` tests that catch bugs; and **Runtime → Restart session and run all** on this notebook.

## Finish

- [ ] I can explain the difference between `print` and `return`, and what `None` means.
- [ ] My `minibuses_needed` rounds up, and my tests still pass with 15 seats (Try it 6 and 7).
- [ ] `rules.py` and `records.py` are in my folder, all five checks pass, and `bridge.show()` shows them as mine.
- [ ] My decision log records my `URGENT_HOURS` choice and its tradeoff.
- [ ] I found three problems in Version A, and I can say which line breaks the AI's `open_portions`.
- [ ] I can say how to ask an AI for code with a contract, and how I would check its answer.

### Research 3 · Checking input (5%)

*Set after the midterm.* Why do systems check input **before** they save it? Choose one form you have used: a university registration form, a bus or ticket booking, a mobile top-up, an online shop checkout, a club sign-up.

For **two fields** of that form, give:

1. the **allowed values** (for example "a whole number from 1 to 10");
2. an **invalid** example;
3. the **message** the form shows, or the message it *should* show;
4. what must **stay unchanged** when the input is bad.

Use invented examples, never your own real personal data. Use at least one source and name it. If you use an AI assistant, show one place where you checked or corrected it. Bring one of your fields to the **Week 5 class warm-up**. Submit by email to submit.progmis@gmail.com; the deadline and file name will be announced in class.

**Next week:** you add `data_model.py`, the check at the door, and watch nine messy rows turn green or red with your own messages.

⬅ **Previous:** [Week 3 · Records and data](Week_03.ipynb) · **Session 3:** Midterm (see the Midterm menu tab) · **Next:** [Week 5 · Checking input](Week_05.ipynb) ➡